# 📈 Regressão Linear, Séries Temporais e Diagnósticos de Qualidade (ANOVA/Teste t)

Este notebook demonstra a aplicação prática de modelos lineares para:
1. **Previsão de Séries Temporais**: Utilizando janelamento temporal e regressão Ridge.
2. **Diagnósticos de Regressão e ANOVA**: Replicando os testes estatísticos de utilidade (t-test) e análise de variabilidade apresentados em aula.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t
from src import (
    make_windows,
    CustomRidgeRegression,
    CustomLinearRegression,
    train_test_split,
    mean_squared_error,
    r2_score,
    teste_utilidade_regressao,
    analise_qualidade_regressao
)

## Parte 1: Previsão de Série Temporal Autoregressiva
Modelamos e prevemos uma série temporal com tendência, sazonalidade e ruído gaussiano.

In [ ]:
np.random.seed(42)
t_series = np.arange(144)  # 12 anos de dados mensais
trend = 0.15 * t_series
seasonality = 10 * np.sin(2 * np.pi * t_series / 12)
noise = np.random.normal(0, 1.5, size=144)
series = 20 + trend + seasonality + noise

plt.figure(figsize=(10, 5))
plt.plot(t_series, series, label='Série Temporal Real', color='tab:blue')
plt.title('Série Temporal Sintética')
plt.xlabel('Tempo')
plt.ylabel('Valor')
plt.legend()
plt.grid(True)
plt.show()

### 1. Preparação dos Dados (Janelas Deslizantes)
Usamos `make_windows` com tamanho de janela igual a 12 (um ciclo anual) para prever o próximo mês.

In [ ]:
window_size = 12
X, y = make_windows(series, window_size)
print('Shape de X:', X.shape)
print('Shape de y:', y.shape)

### 2. Divisão Treino/Teste Sequencial

In [ ]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Treino: {X_train.shape[0]} amostras, Teste: {X_test.shape[0]} amostras')

### 3. Treinamento da Regressão Ridge Customizada

In [ ]:
ridge = CustomRidgeRegression(alpha=1.0, fit_intercept=True)
ridge.fit(X_train, y_train)

print('Coeficientes ajustados:', ridge.coef_)
print('Intercepto ajustado:', ridge.intercept_)

### 4. Previsão e Avaliação

In [ ]:
y_pred = ridge.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Erro Quadrático Médio (MSE): {mse:.4f}')
print(f'Coeficiente de Determinação (R²): {r2:.4f}')

### 5. Visualização das Previsões

In [ ]:
plt.figure(figsize=(12, 6))
test_range = np.arange(split_idx + window_size, len(series))
plt.plot(t_series, series, label='Histórico Real', color='tab:blue')
plt.plot(test_range, y_pred, label='Previsão (Custom Ridge)', color='tab:red', linestyle='--')
plt.axvline(x=split_idx + window_size, color='gray', linestyle=':', label='Início do Teste')
plt.title('Previsão de Série Temporal com Regressão Regularizada Ridge do Zero')
plt.xlabel('Tempo')
plt.ylabel('Valor')
plt.legend()
plt.grid(True)
plt.show()

## Parte 2: Diagnósticos de Regressão Linear Simples e ANOVA (Exemplos de Aula)
Nesta seção, realizamos os testes analíticos de hipóteses e análises de qualidade de regressão linear a partir de dados simulados de habitações.

In [ ]:
# Geração de dados de habitação: sqft_living e price
np.random.seed(42)
n_samples = 100
sqft_living = np.random.normal(loc=2000, scale=500, size=n_samples)
price = 50000 + 180 * sqft_living + np.random.normal(loc=0, scale=30000, size=n_samples)

df_house = pd.DataFrame({'sqft_living': sqft_living, 'price': price})
df_house.head()

### 1. Teste de Utilidade de Regressão Simples
Calculamos a estatística $t$ e o p-valor bicaudal para testar a hipótese nula $H_0: \beta_1 = 0$.

In [ ]:
beta_0, beta_1, ep_beta_1, t_stat, p_value = teste_utilidade_regressao(
    df_house['sqft_living'].values,
    df_house['price'].values,
    alpha=0.05
)

tabela_t = pd.DataFrame({
    'Valor Estimado': [beta_0, beta_1],
    'Erro Padrão': [np.nan, ep_beta_1],
    't-stat': [np.nan, t_stat],
    'p-valor': [np.nan, p_value]
}, index=['Intercepto (beta_0)', 'Área Habitável (beta_1)'])

print('Tabela de Teste de Utilidade:')
tabela_t

### 2. Tabela ANOVA da Regressão Simples

In [ ]:
_, _, SQE, STQT, SQR, R2 = analise_qualidade_regressao(
    df_house['sqft_living'].values,
    df_house['price'].values
)

tabela_anova = pd.DataFrame({
    'Soma dos Quadrados': [SQR, SQE, STQT]
}, index=['Regressão (SQR)', 'Erros/Resíduos (SQE)', 'Total (STQT)'])

print(f'R² do Modelo: {R2:.4f}')
tabela_anova

### 3. Diagnósticos de Regressão Múltipla com Tabela e Dispersão (Função de Aula)
Apresentamos a função de regressão múltipla de aula executada no nosso ecossistema customizado.

In [ ]:
def analise_qualidade_regressao_multipla_com_tabela_e_scatter(X, Y, alpha=0.05):
    n = X.shape[0]
    X_aug = np.column_stack((np.ones(n), X))
    p = X_aug.shape[1]
    
    beta = np.linalg.inv(X_aug.T @ X_aug) @ (X_aug.T @ Y)
    Y_hat = X_aug @ beta
    Y_bar = np.mean(Y)
    residuo = Y - Y_hat
    
    SQE = np.sum(residuo**2)
    STQT = np.sum((Y - Y_bar)**2)
    SQR = STQT - SQE
    R2 = SQR / STQT
    
    s2 = SQE / (n - p)
    var_beta = np.diag(s2 * np.linalg.inv(X_aug.T @ X_aug))
    se_beta = np.sqrt(var_beta)
    
    t_stats = beta / se_beta
    p_valores = 2 * (1 - t.cdf(np.abs(t_stats), df=n - p))
    
    nomes = ['Intercepto'] + [f'X{j}' for j in range(1, p)]
    tabela_resultados = pd.DataFrame({
        'Coeficiente (β̂)': beta,
        'Erro Padrão': se_beta,
        't-valor': t_stats,
        'p-valor': p_valores
    }, index=nomes)
    
    print(f'SQE = {SQE:.4f} | STQT = {STQT:.4f} | SQR = {SQR:.4f} | R² = {R2:.4f}')
    
    # Gráficos Y vs Xj
    num_cols = X.shape[1]
    fig, axs = plt.subplots(1, num_cols, figsize=(5 * num_cols, 4))
    if num_cols == 1:
        axs = [axs]
    for j in range(num_cols):
        axs[j].scatter(X[:, j], Y, color='black')
        axs[j].set_title(f'Y vs X{j+1}')
        axs[j].set_xlabel(f'X{j+1}')
        axs[j].set_ylabel('Y')
        axs[j].grid(True)
    plt.tight_layout()
    plt.show()
    
    # Gráfico de resíduos
    plt.figure(figsize=(6, 4))
    plt.scatter(Y_hat, residuo, color='blue')
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel('Valores Ajustados')
    plt.ylabel('Resíduos')
    plt.title('Resíduos vs Valores Ajustados')
    plt.grid(True)
    plt.show()
    
    return beta, residuo, tabela_resultados

# Regressão múltipla usando sqft_living e número de quartos (bedrooms)
bedrooms = np.random.randint(2, 6, size=n_samples)
X_multi = np.column_stack((sqft_living, bedrooms))
y_multi = price + 15000 * bedrooms

beta, residuo, tabela = analise_qualidade_regressao_multipla_com_tabela_e_scatter(X_multi, y_multi)
tabela